In [1]:
# ==============================================================================
# VORTEX-IN-CELL (VIC) FFT Fast Poisson/Biot-Savart Solver via CuPy
# O(N log N) direct GPU convolution replacing the Slow O(N^2) or CPU FMM
# ==============================================================================
import cupy as cp

def build_kernel_fft(nx, ny, dx, blob_radius):
    """
    Precomputes the zero-padded 2D Fast Fourier Transforms of the Biot-Savart
    velocity kernels (K_x and K_y) for a Cartesian Grid.
    """
    # Double the domain for zero-padded convolution
    NX, NY = 2 * nx, 2 * ny
    x_grid = cp.arange(NX) * dx
    y_grid = cp.arange(NY) * dx

    # Shift to center the kernels to properly wrap periodically
    x_grid = cp.where(x_grid > nx * dx, x_grid - NX * dx, x_grid)
    y_grid = cp.where(y_grid > ny * dx, y_grid - NY * dx, y_grid)

    X, Y = cp.meshgrid(x_grid, y_grid, indexing='ij')
    r2 = X**2 + Y**2 + blob_radius**2

    # Biot-Savart Kernel evaluation
    K_u = -Y / (2.0 * cp.pi * r2)
    K_v =  X / (2.0 * cp.pi * r2)

    # Deal with singularity safely
    K_u[0, 0] = 0.0
    K_v[0, 0] = 0.0

    # Pre-calculate their transforms
    K_u_hat = cp.fft.fft2(K_u)
    K_v_hat = cp.fft.fft2(K_v)

    return K_u_hat, K_v_hat

def fft_velocity_field(omega_grid, K_u_hat, K_v_hat):
    """
    Computes velocity field on the grid via FFT convolution of vorticity
    """
    nx, ny = omega_grid.shape
    NX, NY = 2 * nx, 2 * ny

    # Zero pad vorticity grid to 2N x 2N
    omega_pad = cp.zeros((NX, NY), dtype=cp.float64)
    omega_pad[:nx, :ny] = omega_grid

    # Forward Transform
    omega_hat = cp.fft.fft2(omega_pad)

    # Multiply in frequency domain (Convolution Theorem)
    U_hat = K_u_hat * omega_hat
    V_hat = K_v_hat * omega_hat

    # Inverse Transform and remove padding
    U_grid = cp.fft.ifft2(U_hat).real[:nx, :ny]
    V_grid = cp.fft.ifft2(V_hat).real[:nx, :ny]

    return U_grid, V_grid

def interpolate_velocities_to_particles(px, py, U_grid, V_grid, x_min, y_min, dx, nx, ny):
    """
    Bilinear interpolation from grid velocities back to continuous particle coordinates.
    """
    # Sub-index in grid coordinates
    idx_x = (px - x_min) / dx
    idx_y = (py - y_min) / dx

    i = cp.floor(idx_x).astype(cp.int32)
    j = cp.floor(idx_y).astype(cp.int32)

    # Bound to valid indices
    i = cp.clip(i, 0, nx - 2)
    j = cp.clip(j, 0, ny - 2)

    wx = idx_x - i
    wy = idx_y - j

    w00 = (1.0 - wx) * (1.0 - wy)
    w10 = wx * (1.0 - wy)
    w01 = (1.0 - wx) * wy
    w11 = wx * wy

    u_p = (U_grid[i, j]*w00 + U_grid[i+1, j]*w10 +
           U_grid[i, j+1]*w01 + U_grid[i+1, j+1]*w11)

    v_p = (V_grid[i, j]*w00 + V_grid[i+1, j]*w10 +
           V_grid[i, j+1]*w01 + V_grid[i+1, j+1]*w11)

    return u_p, v_p


def build_omega_grid(px, py, gamma, x_min, y_min, dx, nx, ny):
    """
    Build a CuPy vorticity grid from particle circulation for FFT velocity convolution.
    """
    i = cp.floor((px - x_min) / dx).astype(cp.int32)
    j = cp.floor((py - y_min) / dx).astype(cp.int32)
    i = cp.clip(i, 0, nx - 1)
    j = cp.clip(j, 0, ny - 1)
    omega_grid = cp.zeros((nx, ny), dtype=cp.float64)
    cp.add.at(omega_grid, (i, j), gamma)
    return omega_grid


In [ ]:
import numpy as np
from numba import jit, prange

# -------------------------------------------------------------------------
# CPU-Accelerated Fast Multipole Method (Treecode) via Numba
# Ported exactly from original Fortran / pyVPM algorithm
# Multi-threading enabled via prange
# -------------------------------------------------------------------------

class Panel:
    def __init__(self, label=0, level=0, center=None, side_length=0.0):
        self.label = label
        self.level = level
        self.first_arrival = -1
        self.last_arrival = -1
        self.first_departure = -1
        self.last_departure = -1
        self.binary_pos = np.zeros(2, dtype=int)
        self.center = np.zeros(2, dtype=float) if center is None else np.array(center, dtype=float)
        self.side_length = float(side_length)
        self.ak = None
        self.bk = None
        self.children = []

class QuadTree:
    def __init__(self, max_level=5, ext_dom=0.0):
        self.max_level = max_level
        self.ext_dom = ext_dom
        self.panels = []
        self.binary_identification = np.array([[0,0], [0,1], [1,0], [1,1]], dtype=int)

    def decomposition_init(self, arrival_pos, departure_pos):
        min_pos_dept = np.min(departure_pos, axis=1) if departure_pos.shape[1] > 0 else np.array([0., 0.])
        max_pos_dept = np.max(departure_pos, axis=1) if departure_pos.shape[1] > 0 else np.array([0., 0.])
        min_pos_arr = np.min(arrival_pos, axis=1) if arrival_pos.shape[1] > 0 else min_pos_dept
        max_pos_arr = np.max(arrival_pos, axis=1) if arrival_pos.shape[1] > 0 else max_pos_dept

        min_pos = np.minimum(min_pos_dept, min_pos_arr)
        max_pos = np.maximum(max_pos_dept, max_pos_arr)

        length_dom_x = max_pos[0] - min_pos[0]
        length_dom_y = max_pos[1] - min_pos[1]

        if length_dom_x > length_dom_y:
            min_pos[1] -= (length_dom_x - length_dom_y) * 0.5
        elif length_dom_x < length_dom_y:
            min_pos[0] -= (length_dom_y - length_dom_x) * 0.5

        length_dom = max(length_dom_x, length_dom_y) * (1.0 + self.ext_dom)
        min_pos -= self.ext_dom * 0.5 * length_dom

        root_panel = Panel(label=0, level=0, center=min_pos + length_dom * 0.5, side_length=length_dom)
        self.panels = [root_panel]
        return root_panel

    def subdivide_panel(self, parent_panel, arrival_pos, departure_pos, arr_indices, dep_indices):
        sl = parent_panel.side_length / 4.0
        cc = parent_panel.center
        centers = [
            np.array([cc[0] - sl, cc[1] + sl]), np.array([cc[0] - sl, cc[1] - sl]),
            np.array([cc[0] + sl, cc[1] + sl]), np.array([cc[0] + sl, cc[1] - sl])
        ]

        children = []
        for i in range(4):
            child = Panel(label=len(self.panels)+i, level=parent_panel.level+1, center=centers[i], side_length=parent_panel.side_length/2.0)
            child.binary_pos = parent_panel.binary_pos * 2 + self.binary_identification[i]
            children.append(child)

        parent_panel.children = children
        arr_child_indices = [np.array([], dtype=int) for _ in range(4)]
        dep_child_indices = [np.array([], dtype=int) for _ in range(4)]

        if len(arr_indices) > 0:
            pts = arrival_pos[:, arr_indices]
            dx, dy = pts[0,:] - cc[0], -pts[1,:] + cc[1]
            bx, by = ((1.0 + np.where(dx >= 0, 1.0, -1.0))/2).astype(int), ((1.0 + np.where(dy >= 0, 1.0, -1.0))/2).astype(int)
            idx = bx * 2 + by
            for i in range(4):
                arr_child_indices[i] = arr_indices[idx == i]

        if len(dep_indices) > 0:
            pts = departure_pos[:, dep_indices]
            dx, dy = pts[0,:] - cc[0], -pts[1,:] + cc[1]
            bx, by = ((1.0 + np.where(dx >= 0, 1.0, -1.0))/2).astype(int), ((1.0 + np.where(dy >= 0, 1.0, -1.0))/2).astype(int)
            idx = bx * 2 + by
            for i in range(4):
                dep_child_indices[i] = dep_indices[idx == i]

        return children, arr_child_indices, dep_child_indices

    def build_tree(self, arrival_pos, departure_pos, max_part_per_panel=100):
        root = self.decomposition_init(arrival_pos, departure_pos)
        n_arr = arrival_pos.shape[1] if arrival_pos.ndim == 2 else 0
        n_dep = departure_pos.shape[1] if departure_pos.ndim == 2 else 0
        root.first_arrival, root.last_arrival = (0, n_arr - 1) if n_arr > 0 else (-1, -1)
        root.first_departure, root.last_departure = (0, n_dep - 1) if n_dep > 0 else (-1, -1)

        new_arr_ind, new_dep_ind = [], []

        def _build(panel, a_ind, d_ind):
            if panel.level >= self.max_level or len(d_ind) <= max_part_per_panel:
                if len(a_ind) > 0:
                    panel.first_arrival = len(new_arr_ind)
                    new_arr_ind.extend(a_ind)
                    panel.last_arrival = len(new_arr_ind) - 1
                else:
                    panel.first_arrival, panel.last_arrival = -1, -1

                if len(d_ind) > 0:
                    panel.first_departure = len(new_dep_ind)
                    new_dep_ind.extend(d_ind)
                    panel.last_departure = len(new_dep_ind) - 1
                else:
                    panel.first_departure, panel.last_departure = -1, -1
                return

            children, a_sub, d_sub = self.subdivide_panel(panel, arrival_pos, departure_pos, a_ind, d_ind)
            self.panels.extend(children)
            for i, c in enumerate(children):
                _build(c, a_sub[i], d_sub[i])

            v_a = [c for c in children if c.first_arrival != -1]
            panel.first_arrival, panel.last_arrival = (v_a[0].first_arrival, v_a[-1].last_arrival) if v_a else (-1, -1)

            v_d = [c for c in children if c.first_departure != -1]
            panel.first_departure, panel.last_departure = (v_d[0].first_departure, v_d[-1].last_departure) if v_d else (-1, -1)

        _build(root, np.arange(n_arr), np.arange(n_dep))
        return np.array(new_arr_ind, dtype=int), np.array(new_dep_ind, dtype=int)


@jit(nopython=True, fastmath=True, parallel=True)
def direct_velocity_kernel(target_x, target_y, source_x, source_y, gamma, blob_radius):
    nt, ns = len(target_x), len(source_x)
    u, v = np.zeros(nt), np.zeros(nt)
    r_sq = blob_radius * blob_radius

    for i in prange(nt):
        tx, ty = target_x[i], target_y[i]
        u_a, v_a = 0.0, 0.0
        for j in range(ns):
            dx, dy = tx - source_x[j], ty - source_y[j]
            d2 = dx*dx + dy*dy + r_sq

            if d2 > 1e-14:
                val = gamma[j] / (2.0 * np.pi * d2)
                u_a -= val * dy
                v_a += val * dx
        u[i], v[i] = u_a, v_a

    return u, v

@jit(nopython=True, fastmath=True)
def compute_panel_multipoles(cx, cy, p_x, p_y, p_gamma, p_start, p_end, n_terms=10):
    ak, bk = np.zeros(n_terms), np.zeros(n_terms)

    for j in range(p_start, p_end):
        g = p_gamma[j] / (2.0 * np.pi)
        dx, dy = p_x[j] - cx, p_y[j] - cy

        bk[0] -= g
        zx, zy = 1.0, 0.0

        for k in range(1, n_terms):
            zx, zy = zx * dx - zy * dy, zx * dy + zy * dx
            ak[k] += g * zy
            bk[k] -= g * zx

    return ak, bk

@jit(nopython=True, fastmath=True)
def evaluate_multipole_velocity(tx, ty, cx, cy, ak, bk, n_terms=10):
    dx, dy = tx - cx, ty - cy
    r2 = dx*dx + dy*dy
    if r2 < 1e-14: return 0.0, 0.0

    u, v = 0.0, 0.0
    tzx, tzy = dx / r2, -dy / r2
    termx, termy = tzx, tzy

    for k in range(n_terms):
        u += ak[k] * termx - bk[k] * termy
        v -= ak[k] * termy + bk[k] * termx
        termx, termy = termx * tzx - termy * tzy, termx * tzy + termy * tzx

    return u, v

def fmm_velocity(targ_x, targ_y, src_x, src_y, gamma, blob_radius, max_particles=50, max_level=4, n_terms=10, theta=0.5, **kwargs):
    """
    Original pyVPM Fast Multipole Method (Treecode approach), run on the CPU.
    """
    if len(targ_x) == 0 or len(src_x) == 0:
        return np.zeros_like(targ_x), np.zeros_like(targ_y)

    qt = QuadTree(max_level=max_level)
    pos_arr = np.vstack((targ_x, targ_y))
    pos_dep = np.vstack((src_x, src_y))
    arr_idx, dep_idx = qt.build_tree(pos_arr, pos_dep, max_part_per_panel=max_particles)

    # Sort sources based on Tree
    x_d = src_x[dep_idx]
    y_d = src_y[dep_idx]
    g_d = gamma[dep_idx]

    # Sort targets based on Tree
    x_a = targ_x[arr_idx]
    y_a = targ_y[arr_idx]
    u_a = np.zeros_like(x_a)
    v_a = np.zeros_like(y_a)

    # Upward pass: Pre-compute multipole moments for all nodes with sources
    for p in qt.panels:
        if p.first_departure != -1:
            p.ak, p.bk = compute_panel_multipoles(p.center[0], p.center[1], x_d, y_d, g_d, p.first_departure, p.last_departure + 1, n_terms)

    # Downward pass: Evaluate velocities combining Near-Field and Far-Field expansions
    for p in qt.panels:
        # Only evaluate targets located inside empty leaf nodes
        if p.first_arrival == -1 or len(p.children) > 0:
            continue

        astart, aend = p.first_arrival, p.last_arrival + 1
        t_x, t_y = x_a[astart:aend], y_a[astart:aend]

        def _traverse(node):
            if node.first_departure == -1: return
            dist = np.sqrt((p.center[0] - node.center[0])**2 + (p.center[1] - node.center[1])**2)

            # Multipole Acceptance Criterion (MAC)
            if node.level > 0 and (node.side_length / dist < theta if dist > 0 else False):
                # Far-field
                for i in range(len(t_x)):
                    du, dv = evaluate_multipole_velocity(t_x[i], t_y[i], node.center[0], node.center[1], node.ak, node.bk, n_terms)
                    u_a[astart + i] += du
                    v_a[astart + i] += dv
            elif len(node.children) == 0:
                # Near-field
                dstart, dend = node.first_departure, node.last_departure + 1
                du, dv = direct_velocity_kernel(t_x, t_y, x_d[dstart:dend], y_d[dstart:dend], g_d[dstart:dend], blob_radius)
                u_a[astart:aend] += du
                v_a[astart:aend] += dv
            else:
                for c in node.children: _traverse(c)

        _traverse(qt.panels[0])

    # Re-map targets back to their original array indices
    u = np.zeros_like(u_a)
    v = np.zeros_like(v_a)
    u[arr_idx] = u_a
    v[arr_idx] = v_a

    return u, v

In [3]:
# Mount Google Drive to save results directly and prevent data loss
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BASE = '/content/drive/MyDrive/VPM_Simulation_Results'
    import os
    os.makedirs(DRIVE_BASE, exist_ok=True)
    print(f'Drive mounted. Results will be saved to {DRIVE_BASE}')
except ImportError:
    print('Not running in Colab, saving locally.')
    DRIVE_BASE = '.'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted. Results will be saved to /content/drive/MyDrive/VPM_Simulation_Results


# 6-Cylinder GPU Accelerated Solver for PINNs
This notebook runs the 2D Discrete Vortex Method exclusively on the Colab Nvidia GPU using `cupy`.

**Instructions:**
1. Go to `Runtime` -> `Change runtime type` and ensure **T4 GPU** is selected.
2. Ensure your `dom1.dat` geometry file is located at `/content/drive/MyDrive/vpm/dom1.dat`.
3. Run all cells below.

In [4]:
# Install specific cupy version if not natively active in Colab
import cupy as cp

# Safely print the active GPU device name
import cupy.cuda.runtime as runtime
device_props = runtime.getDeviceProperties(0)
gpu_name = device_props['name'].decode('utf-8')
print(f"CuPy Successfully Loaded. Active Device: {gpu_name}")

import numpy as np
import math
import os

CuPy Successfully Loaded. Active Device: NVIDIA L4


In [5]:
# ==============================================================================
# GPU NATIVE KERNELS
# ==============================================================================

diffusion_kernel_code = '''
extern "C" __global__
void remesh_particles(
    const double* p_x, const double* p_y, const double* p_gamma,
    double* mesh_circ,
    double x_min, double y_min, double dx, int nx, int ny,
    double rdiff_sq, double diff_coeff, int nodes_r, int n_particles)
{
    int tid = blockDim.x * blockIdx.x + threadIdx.x;
    if (tid >= n_particles) return;

    double px = p_x[tid];
    double py = p_y[tid];
    double gamma = p_gamma[tid];

    int i_center = round((px - x_min) / dx);
    int j_center = round((py - y_min) / dx);

    int i_start = max(0, i_center - nodes_r);
    int i_end = min(nx, i_center + nodes_r + 1);
    int j_start = max(0, j_center - nodes_r);
    int j_end = min(ny, j_center + nodes_r + 1);

    double total_weight = 0.0;

    for (int i = i_start; i < i_end; ++i) {
        double node_x = x_min + i * dx;
        double dx_val = node_x - px;
        double dx_sq = dx_val * dx_val;
        if (dx_sq > rdiff_sq) continue;

        for (int j = j_start; j < j_end; ++j) {
            double node_y = y_min + j * dx;
            double dy_val = node_y - py;
            double dy_sq = dy_val * dy_val;
            double dist2 = dx_sq + dy_sq;

            if (dist2 <= rdiff_sq) {
                total_weight += exp(-dist2 * diff_coeff);
            }
        }
    }

    if (total_weight < 1e-20) {
        if (i_center >= 0 && i_center < nx && j_center >= 0 && j_center < ny) {
            atomicAdd(&mesh_circ[i_center * ny + j_center], gamma);
        }
        return;
    }

    double inv_total = 1.0 / total_weight;

    for (int i = i_start; i < i_end; ++i) {
        double node_x = x_min + i * dx;
        double dx_val = node_x - px;
        double dx_sq = dx_val * dx_val;
        if (dx_sq > rdiff_sq) continue;

        for (int j = j_start; j < j_end; ++j) {
            double node_y = y_min + j * dx;
            double dy_val = node_y - py;
            double dy_sq = dy_val * dy_val;
            double dist2 = dx_sq + dy_sq;

            if (dist2 <= rdiff_sq) {
                double weight = exp(-dist2 * diff_coeff);
                atomicAdd(&mesh_circ[i * ny + j], weight * inv_total * gamma);
            }
        }
    }
}
'''

remesh_cuda = cp.RawKernel(diffusion_kernel_code, 'remesh_particles')

def compute_velocity_gpu(target_x, target_y, source_x, source_y, source_gamma, blob_radius, chunk_size=20000):
    u_ind = cp.zeros_like(target_x)
    v_ind = cp.zeros_like(target_x)
    n_targets = len(target_x)
    n_sources = len(source_x)
    r_sq_core = blob_radius**2
    factor_pi = 2.0 * cp.pi

    for i in range(0, n_targets, chunk_size):
        end_i = min(i + chunk_size, n_targets)
        tx = target_x[i:end_i][:, None]
        ty = target_y[i:end_i][:, None]
        for j in range(0, n_sources, chunk_size):
            end_j = min(j + chunk_size, n_sources)
            sx = source_x[j:end_j][None, :]
            sy = source_y[j:end_j][None, :]
            sg = source_gamma[j:end_j][None, :]

            dx = tx - sx
            dy = ty - sy
            d2 = dx**2 + dy**2

            denom = d2 + r_sq_core
            val = cp.where(denom > 1e-20, sg / (factor_pi * denom), 0.0)

            u_ind[i:end_i] += cp.sum(-val * dy, axis=1)
            v_ind[i:end_i] += cp.sum( val * dx, axis=1)
    return u_ind, v_ind

def redistribute_and_remesh_gpu(x, y, gamma, cur_domain):
    if cur_domain['vis_cin'] <= 1e-12: return x, y, gamma
    nx = cur_domain['nx']
    ny = cur_domain['ny']
    num_particles = len(x)
    mesh_circ = cp.zeros(nx * ny, dtype=cp.float64)

    if num_particles > 0:
        nodes_r = int(math.ceil(cur_domain['rdiff'] / cur_domain['dx'])) + 1
        threads_per_block = 1024
        blocks = (num_particles + threads_per_block - 1) // threads_per_block
        remesh_cuda((blocks,), (threads_per_block,), (
            x, y, gamma, mesh_circ,
            cur_domain['x_min'], cur_domain['y_min'], cur_domain['dx'],
            nx, ny, cur_domain['rdiff_sq'], cur_domain['diff_coeff'], nodes_r, num_particles
        ))

    mesh_circ = mesh_circ.reshape((nx, ny))
    active_mask = cp.abs(mesh_circ) > cur_domain['diff_cut']
    indices_i, indices_j = cp.where(active_mask)
    new_gamma = mesh_circ[active_mask]
    new_x = cur_domain['x_min'] + indices_i * cur_domain['dx']
    new_y = cur_domain['y_min'] + indices_j * cur_domain['dx']
    return new_x, new_y, new_gamma

In [ ]:
# ===============================================================================
# CORE PHYSICS ENGINE
# ===============================================================================

class BodyGPU:
    def __init__(self, filename):
        px, py = [], []
        with open(filename, 'r') as f:
            lines = f.readlines()
        in_points = False
        n_points = 0
        read_points = 0
        for line in lines:
            line = line.strip()
            if not line or line.startswith('!'): continue
            if 'nCpoints_of_Chunk_1' in line:
                try:
                    n_points = int(line.split()[0])
                except ValueError:
                    n_points = int(line.split()[1])
                in_points = True
                continue
            if in_points:
                parts = line.split()
                try:
                    px.append(float(parts[0]))
                    py.append(float(parts[1]))
                    read_points += 1
                except ValueError:
                    pass
                if read_points == n_points: break

        # Remove the closing duplicate point if present to avoid ZeroDivisionError
        if px and py and len(px) > 1 and px[0] == px[-1] and py[0] == py[-1]:
            px, py = px[:-1], py[:-1]
        self.points = np.vstack((np.array(px), np.array(py)))

    def translate_and_compute(self, dx_offset, dy_offset):
        self.points[0, :] += dx_offset
        self.points[1, :] += dy_offset
        n = self.points.shape[1]
        x, y = self.points[0, :], self.points[1, :]
        lengths, mid_x, mid_y = np.zeros(n), np.zeros(n), np.zeros(n)
        tx, ty, nx_n, ny_n = np.zeros(n), np.zeros(n), np.zeros(n), np.zeros(n)

        for i in range(n):
            j = (i + 1) % n
            dx_p, dy_p = x[j] - x[i], y[j] - y[i]
            length = math.sqrt(dx_p**2 + dy_p**2)
            lengths[i] = length
            mid_x[i], mid_y[i] = x[i] + 0.5 * dx_p, y[i] + 0.5 * dy_p
            tx[i], ty[i] = dx_p / length, dy_p / length
            nx_n[i], ny_n[i] = dy_p / length, -dx_p / length

        self.mid_x_gp = cp.asarray(mid_x)
        self.mid_y_gp = cp.asarray(mid_y)
        self.tx_gp = cp.asarray(tx)
        self.ty_gp = cp.asarray(ty)
        self.nx_gp = cp.asarray(nx_n)
        self.ny_gp = cp.asarray(ny_n)
        self.lengths_gp = cp.asarray(lengths)
 
        self.center_x = self.points[0, :].mean()
        self.center_y = self.points[1, :].mean()
        self.radius = np.mean(np.sqrt((self.points[0, :] - self.center_x)**2 + (self.points[1, :] - self.center_y)**2))

    def kill_inside_particles(self, px_gpu, py_gpu, gamma_gpu):
        dist_sq = (px_gpu - self.center_x)**2 + (py_gpu - self.center_y)**2
        outside = dist_sq > (self.radius**2)
        return px_gpu[outside], py_gpu[outside], gamma_gpu[outside]

class GPU_VPM_Solver:
    def __init__(self, dt, re, dx, R_formation, num_cyl):
        self.dt = dt
        self.u_inf = 1.0
        self.v_inf = 0.0
        self.blob_radius = 1.5 * dx

        viscosity = 1.0 / re
        dVol0 = dx * dx
        Nnode_support = 51.0
        pi_inv = 1.0 / math.pi
        error_exp = 5.0

        rdiff = math.sqrt(Nnode_support * pi_inv * dVol0)
        target_dt_diff = (Nnode_support * pi_inv * dVol0 * (1.0/viscosity) * 0.25) / (error_exp * math.log(10.0))
        cdt = max(1, int(round(target_dt_diff / dt)))
        dt_diff = cdt * dt
        diff_coeff = 0.25 * (1.0/viscosity) / dt_diff
        diff_cut = 0.25 * pi_inv * (1.0/viscosity) / dt_diff * math.exp(- (rdiff**2) * diff_coeff) * (dVol0**2)

        #x_min_m, x_max_m, y_min_m, y_max_m = -15.0 - rdiff*1.5, 30.0 + rdiff*1.5, -15.0 - rdiff*1.5, 15.0 + rdiff*1.5
        x_min_m, x_max_m, y_min_m, y_max_m = -15.0 - rdiff*1.5, 45.0 + rdiff*1.5, -30.0 - rdiff*1.5, 30.0 + rdiff*1.5
        self.domain = {
            'x_min': x_min_m, 'x_max': x_max_m, 'y_min': y_min_m, 'y_max': y_max_m,
            'nx': int((x_max_m - x_min_m) / dx) + 1, 'ny': int((y_max_m - y_min_m) / dx) + 1,
            'dx': dx, 'rdiff': rdiff, 'rdiff_sq': rdiff**2, 'diff_coeff': diff_coeff,
            'diff_cut': diff_cut, 'vis_cin': viscosity
        }

        self.K_u_hat, self.K_v_hat = build_kernel_fft(self.domain['nx'], self.domain['ny'], self.domain['dx'], self.blob_radius)

        self.x, self.y, self.gamma = cp.array([], dtype=cp.float64), cp.array([], dtype=cp.float64), cp.array([], dtype=cp.float64)

        self.bodies = []
        for i, angle in enumerate(np.linspace(0, 2 * np.pi, num_cyl, endpoint=False)):
            cx, cy = R_formation * 1.0 * np.cos(angle), R_formation * 1.0 * np.sin(angle)
            body = BodyGPU('/content/drive/MyDrive/VPM_Simulation_Results/Dom1_300.dat')
            body.translate_and_compute(cx, cy)
            self.bodies.append(body)
        self.time = 0.0
        self.forces = []
        self.diagnostics = []

    def load_state(self, step, folder):
        import glob
        import os
        vtk_file = f"{folder}/step_{step:04d}.vtk"
        if os.path.exists(vtk_file):
            print(f"Restoring internal fluid state from {vtk_file}...")
            with open(vtk_file, 'r') as f:
                lines = f.readlines()
            reading_points, reading_scalars = False, False
            px, py, pgamma = [], [], []
            n_pts = 0
            for line in lines:
                if line.startswith("POINTS"):
                    reading_points = True
                    n_pts = int(line.split()[1])
                    continue
                if reading_points and len(px) < n_pts:
                    pts = line.split()
                    px.append(float(pts[0]))
                    py.append(float(pts[1]))
                    if len(px) == n_pts: reading_points = False
                    continue
                if line.startswith("LOOKUP_TABLE default"):
                    reading_scalars = True
                    continue
                if reading_scalars and len(pgamma) < n_pts:
                    pgamma.append(float(line.strip()))

            self.x = cp.asarray(px, dtype=cp.float64)
            self.y = cp.asarray(py, dtype=cp.float64)
            self.gamma = cp.asarray(pgamma, dtype=cp.float64)

        forces_file = f"{folder}/forces.txt"
        if os.path.exists(forces_file):
            try:
                ld = np.loadtxt(forces_file)
                if ld.ndim == 1: ld = ld.reshape(1, -1)
                self.forces = ld.tolist()
                if len(self.forces) > 0: self.time = self.forces[-1][0]
            except: pass

        diag_file = f"{folder}/diagnostics.txt"
        if os.path.exists(diag_file):
            try:
                ld = np.loadtxt(diag_file)
                if ld.ndim == 1: ld = ld.reshape(1, -1)
                self.diagnostics = ld.tolist()
            except: pass

    def step(self):
        # Apply incoming boundary slip velocity ramp (0.0 to 1.0s smoothing)
        time_ramp = 1.0
        if self.time <= time_ramp:
            arg = self.time * math.pi / time_ramp
            u_scale = max(0.5 * (-math.cos(arg) + 1.0), 1e-5)
        else:
            u_scale = 1.0

        cur_u_inf = self.u_inf * u_scale
        cur_v_inf = self.v_inf * u_scale

        if len(self.x) > 0:
            omega_grid = build_omega_grid(self.x, self.y, self.gamma,
                                          self.domain['x_min'], self.domain['y_min'],
                                          self.domain['dx'], self.domain['nx'], self.domain['ny'])
            U_grid, V_grid = fft_velocity_field(omega_grid, self.K_u_hat, self.K_v_hat)
            u_self, v_self = interpolate_velocities_to_particles(self.x, self.y, U_grid, V_grid,
                                                                 self.domain['x_min'], self.domain['y_min'],
                                                                 self.domain['dx'], self.domain['nx'], self.domain['ny'])
            self.x += (u_self + cur_u_inf) * self.dt
            self.y += (v_self + cur_v_inf) * self.dt
        else:
            U_grid = cp.zeros((self.domain['nx'], self.domain['ny']), dtype=cp.float64)
            V_grid = cp.zeros((self.domain['nx'], self.domain['ny']), dtype=cp.float64)

        new_x, new_y, new_g = [], [], []
        force_row = [self.time + self.dt]
        total_drag = 0.0
        total_lift = 0.0
        rho = 1.0
        nu = self.domain['vis_cin']

        for body in self.bodies:
            if len(self.x) > 0:
                u_ind, v_ind = interpolate_velocities_to_particles(body.mid_x_gp, body.mid_y_gp,
                                                                  U_grid, V_grid,
                                                                  self.domain['x_min'], self.domain['y_min'],
                                                                  self.domain['dx'], self.domain['nx'], self.domain['ny'])
            else:
                u_ind, v_ind = cp.zeros_like(body.mid_x_gp), cp.zeros_like(body.mid_y_gp)

            slip = (cur_u_inf + u_ind) * body.tx_gp + (cur_v_inf + v_ind) * body.ty_gp
            gamma_shed = slip * body.lengths_gp
            # Calculate forces (CUDA vectorised)
            drag_vort = cp.sum(body.tx_gp * slip * body.lengths_gp) * (nu * rho)
            lift_vort = cp.sum(body.ty_gp * slip * body.lengths_gp) * (nu * rho)

            t4 = -0.5 * (slip[1:] + slip[:-1]) / self.dt
            pressure_diffs = rho * body.lengths_gp[1:] * t4
            pressures = cp.cumsum(cp.concatenate([cp.array([0.0]), pressure_diffs]))
            drag_p = cp.sum(-pressures * body.nx_gp * body.lengths_gp)
            lift_p = cp.sum(-pressures * body.ny_gp * body.lengths_gp)

            cyl_drag = float(drag_vort + drag_p)
            cyl_lift = float(lift_vort + lift_p)
            total_drag += cyl_drag
            total_lift += cyl_lift
            force_row.extend([cyl_drag, cyl_lift])
            emit_dist = self.blob_radius * 1.1
            new_x.append(body.mid_x_gp + body.nx_gp * emit_dist)
            new_y.append(body.mid_y_gp + body.ny_gp * emit_dist)
            new_g.append(gamma_shed)

        if len(new_x) > 0:
            self.x = cp.concatenate([self.x] + new_x)
            self.y = cp.concatenate([self.y] + new_y)
            self.gamma = cp.concatenate([self.gamma] + new_g)

        for body in self.bodies:
            self.x, self.y, self.gamma = body.kill_inside_particles(self.x, self.y, self.gamma)

        mask = (self.x >= self.domain['x_min']) & (self.x <= self.domain['x_max']) & (self.y >= self.domain['y_min']) & (self.y <= self.domain['y_max'])
        self.x, self.y, self.gamma = self.x[mask], self.y[mask], self.gamma[mask]
        self.x, self.y, self.gamma = redistribute_and_remesh_gpu(self.x, self.y, self.gamma, self.domain)
        self.time += self.dt
        self.forces.append(force_row)

        # Calculate flow diagnostics
        tot_circ = float(cp.sum(self.gamma))
        enstrophy = float(cp.sum(self.gamma**2))
        self.diagnostics.append((self.time, tot_circ, enstrophy))
    def save_vtk(self, frame, folder="colab_results"):
        os.makedirs(folder, exist_ok=True)
        if len(self.forces) > 0:
            header_cols = ["Time"] + [f"C{i}_Drag C{i}_Lift" for i in range(len(self.bodies))]
            np.savetxt(f"{folder}/forces.txt", np.array(self.forces), header=" ".join(header_cols), fmt="%.6e")
            np.savetxt(f"{folder}/diagnostics.txt", np.array(self.diagnostics), header="Time Total_Circulation Enstrophy_Proxy", fmt="%.6e")
        x_cpu, y_cpu, gamma_cpu = cp.asnumpy(self.x), cp.asnumpy(self.y), cp.asnumpy(self.gamma)
        n_pts = len(x_cpu)
        with open(f"{folder}/step_{frame:04d}.vtk", 'w') as f:
            f.write("# vtk DataFile Version 3.0\nPyVPM Particles\nASCII\nDATASET UNSTRUCTURED_GRID\n")
            f.write(f"POINTS {n_pts} float\n")
            for i in range(n_pts): f.write(f"{x_cpu[i]:.6e} {y_cpu[i]:.6e} 0.0\n")

            f.write(f"\nCELLS {n_pts} {2 * n_pts}\n")
            for i in range(n_pts): f.write(f"1 {i}\n")

            f.write(f"\nCELL_TYPES {n_pts}\n")
            for i in range(n_pts): f.write("1\n")

            f.write(f"\nPOINT_DATA {n_pts}\n")
            f.write("SCALARS Vorticity float 1\nLOOKUP_TABLE default\n")
            for i in range(n_pts): f.write(f"{gamma_cpu[i]:.6e}\n")

### Execute the Simulation
Adjust `R_formation` here to run your specific Anchor Points for the PINN dataset (e.g. 1.5, 3.0, 4.5).

In [ ]:
# Set the Anchor Radius here. Change this for your 3 specific PINN runs!
RADIUS = 0
num_cyl =1
#solver = GPU_VPM_Solver(re=200.0, dt=0.08, dx=0.03, R_formation=RADIUS, num_cyl=1)
#solver = GPU_VPM_Solver(re=200.0, dt=0.04, dx=0.015, R_formation=RADIUS, num_cyl=1)
solver = GPU_VPM_Solver(re=200.0, dt=0.015, dx=0.015, R_formation=RADIUS, num_cyl=1)


total_steps = 3000

import glob
import os
start_step = -1
folder = f"{DRIVE_BASE}/colab_results_R{RADIUS}_{num_cyl}cyl"
os.makedirs(folder, exist_ok=True)
vtks = glob.glob(f"{folder}/step_*.vtk")
if vtks:
    start_step = max([int(os.path.basename(f).split('_')[1].split('.')[0]) for f in vtks])
    solver.load_state(start_step, folder)

print(f"Starting Colab GPU Run: 6 Cylinders at R={RADIUS}. Steps: {total_steps} (Resuming from iteration {start_step if start_step >= 0 else 0})")
for i in range(max(0, start_step + 1), total_steps + 1):
    solver.step()
    if i % 10 == 0:
        solver.save_vtk(i, folder=f"{DRIVE_BASE}/colab_results_R{RADIUS}_{num_cyl}cyl")
        print(f"Step {i:03d} | Particles/Sensors: {len(solver.x)} | Saved VTK")

Starting Colab GPU Run: 6 Cylinders at R=0. Steps: 3000 (Resuming from iteration 0)
Step 000 | Particles/Sensors: 221 | Saved VTK
Step 010 | Particles/Sensors: 2361 | Saved VTK
Step 020 | Particles/Sensors: 4025 | Saved VTK
Step 030 | Particles/Sensors: 5774 | Saved VTK
Step 040 | Particles/Sensors: 7582 | Saved VTK
Step 050 | Particles/Sensors: 9471 | Saved VTK
Step 060 | Particles/Sensors: 11444 | Saved VTK
Step 070 | Particles/Sensors: 13496 | Saved VTK
Step 080 | Particles/Sensors: 15646 | Saved VTK
Step 090 | Particles/Sensors: 17875 | Saved VTK
Step 100 | Particles/Sensors: 20193 | Saved VTK
Step 110 | Particles/Sensors: 22607 | Saved VTK
Step 120 | Particles/Sensors: 25084 | Saved VTK
Step 130 | Particles/Sensors: 27625 | Saved VTK
Step 140 | Particles/Sensors: 30287 | Saved VTK
Step 150 | Particles/Sensors: 32994 | Saved VTK
Step 160 | Particles/Sensors: 35805 | Saved VTK
Step 170 | Particles/Sensors: 38693 | Saved VTK
Step 180 | Particles/Sensors: 41666 | Saved VTK
Step 190 | 

### Download Results
Zips the generated `.vtk` files so you can download them to your laptop for PINN processing.

In [ ]:
!zip -r colab_results_export.zip colab_results_R*
print("Done! You can now download colab_results_export.zip from the files pane.")